# OPRB benchmark: RapidOCR (PP-OCRv6, CUDA)

Local reproducible test-set evaluation using `rapidocr` with `onnxruntime-gpu` (`rec_batch_num=8`).

In [ ]:
import gc, importlib.metadata, json, os, re, subprocess, sys, time, unicodedata
from collections import Counter
from pathlib import Path

def _register_nvidia_dlls():
    nvidia_dir = Path(sys.prefix) / "Lib" / "site-packages" / "nvidia"
    if not nvidia_dir.exists():
        return
    bin_dirs = [str(p) for p in sorted(nvidia_dir.glob("*/bin")) if p.is_dir()]
    if bin_dirs:
        for d in bin_dirs:
            os.add_dll_directory(d)
        os.environ["PATH"] = os.pathsep.join(bin_dirs) + os.pathsep + os.environ.get("PATH", "")

_register_nvidia_dlls()

import onnxruntime as ort
import pandas as pd
from jiwer import cer, wer
from rapidocr import RapidOCR
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = ROOT / "test"
OUTPUT_DIR = ROOT / "result" / "rapidocr_gpu"
REC_BATCH_NUM = 8
SAVE_EVERY = 25
TEXT_LIMIT = 2000
NONTEXT_LIMIT = 1000
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def read_gt(path: Path) -> str:
    tokens = []
    with path.open("r", encoding="utf-8-sig", errors="replace") as f:
        for line in f:
            fields = line.rstrip("\r\n").split("\t")
            if len(fields) >= 2 and fields[0].strip():
                tokens.append(fields[0].strip())
    return " ".join(tokens)

def clean(text: str) -> str:
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", text)).strip().casefold()

def metrics(reference: str, prediction: str) -> dict:
    ref, hyp = clean(reference), clean(prediction)
    if not ref:
        raise ValueError("Empty GT transcript")
    a, b = Counter(ref.split()), Counter(hyp.split())
    matched = sum((a & b).values())
    precision = matched / sum(b.values()) if b else 0.0
    recall = matched / sum(a.values()) if a else 0.0
    return {
        "cer": float(cer(ref, hyp)),
        "wer": float(wer(ref, hyp)),
        "token_precision": precision,
        "token_recall": recall,
        "token_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "gt_words": len(ref.split()),
        "pred_words": len(hyp.split()),
    }

def collect_pairs():
    pairs = []
    for category, limit in (("text", TEXT_LIMIT), ("nontext", NONTEXT_LIMIT)):
        image_dir, gt_dir = DATA_ROOT / category / "img", DATA_ROOT / category / "gt"
        images = {p.stem: p for p in image_dir.glob("*.jpg")}
        gts = {p.stem: p for p in gt_dir.glob("*.txt")}
        common = sorted(images.keys() & gts.keys())[:limit]
        for stem in common:
            parts = stem.split("__", 2)
            pairs.append({
                "category": category,
                "sample_id": stem,
                "image_path": str(images[stem]),
                "gt_path": str(gts[stem]),
                "occlusion_type": parts[0],
                "occlusion_density": parts[1] if len(parts) > 1 else "unknown",
            })
    return pairs

pairs = collect_pairs()
print(f"Pairs: {len(pairs):,} | onnxruntime={ort.__version__}")

In [ ]:
def create_ocr(rec_batch_num: int) -> RapidOCR:
    return RapidOCR(
        params={
            "Global.log_level": "warning",
            "EngineConfig.onnxruntime.use_cuda": True,
            "EngineConfig.onnxruntime.use_dml": False,
            "EngineConfig.onnxruntime.cuda_ep_cfg.arena_extend_strategy": "kSameAsRequested",
            "EngineConfig.onnxruntime.cuda_ep_cfg.gpu_mem_limit": 4 * 1024**3,
            "EngineConfig.onnxruntime.cuda_ep_cfg.cudnn_conv_algo_search": "HEURISTIC",
            "Rec.rec_batch_num": rec_batch_num,
        }
    )

rapidocr_ver = importlib.metadata.version("rapidocr")
model_id = f"RapidOCR {rapidocr_ver} (PP-OCRv6 onnxruntime CUDA, rec_batch_num={REC_BATCH_NUM})"
ocr = create_ocr(REC_BATCH_NUM)

results_csv = OUTPUT_DIR / "per_image_results.csv"
rows = []
t0 = time.perf_counter()

for idx, sample in enumerate(tqdm(pairs, desc=f"RapidOCR (GPU bs={REC_BATCH_NUM})"), 1):
    row = {**sample, "model_id": model_id}
    try:
        start = time.perf_counter()
        out = ocr(sample["image_path"])
        latency = time.perf_counter() - start
        lines = out.txts or ()
        prediction = " ".join(str(t).strip() for t in lines if str(t).strip()).strip()
        reference = read_gt(Path(sample["gt_path"]))
        row.update({
            "status": "ok",
            "prediction": prediction,
            "ground_truth": reference,
            "latency_sec": latency,
            "generated_tokens": 0,
            "possibly_truncated": False,
            **metrics(reference, prediction),
        })
    except Exception as exc:
        row.update({"status": "error", "error": repr(exc)})
    rows.append(row)
    if idx % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(results_csv, index=False)
        gc.collect()

pd.DataFrame(rows).to_csv(results_csv, index=False)
ok = pd.DataFrame(rows)
ok = ok.loc[ok["status"] == "ok"]
summary = {
    "model": model_id,
    "dataset": "kpurkayastha/OPRB test",
    "total_expected": len(pairs),
    "n_success": int(len(ok)),
    "n_errors": int(len(rows) - len(ok)),
    "mean_latency_sec": float(ok["latency_sec"].mean()),
    "mean_cer": float(ok["cer"].mean()),
    "mean_wer": float(ok["wer"].mean()),
    "mean_token_f1": float(ok["token_f1"].mean()),
    "runtime_sec_this_run": time.perf_counter() - t0,
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))